# 💰 Getaround — Optimisation des Prix par Machine Learning

**Objectif :** Entraîner un modèle de régression pour suggérer un prix journalier optimal aux propriétaires.

Le modèle sera exposé via une **API FastAPI** avec endpoint `/predict`.

---

## 1. 📦 Imports & Chargement

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings, joblib
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

try:
    import mlflow
    import mlflow.sklearn
    MLFLOW = True
    mlflow.set_experiment('getaround-pricing')
    print('✅ MLflow activé')
except ImportError:
    MLFLOW = False
    print('ℹ️ MLflow non installé — tracking désactivé')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
SEED = 42

df = pd.read_csv('get_around_pricing_project.csv', index_col=0)
print(f'Dataset : {df.shape[0]:,} lignes x {df.shape[1]} colonnes')
df.head()

## 2. 🔍 Exploration des Données (EDA)

In [ ]:
print('Types et valeurs manquantes :')
print(df.dtypes)
print('\nValeurs manquantes :', df.isnull().sum().sum())
print('\nStatistiques descriptives :')
df.describe().round(1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df['rental_price_per_day'], bins=50, color='#3498db', edgecolor='white', alpha=0.8)
axes[0].axvline(df['rental_price_per_day'].mean(), color='#e74c3c', linestyle='--', lw=2,
                label=f"Moyenne : {df['rental_price_per_day'].mean():.0f}€")
axes[0].axvline(df['rental_price_per_day'].median(), color='#f39c12', linestyle='--', lw=2,
                label=f"Médiane : {df['rental_price_per_day'].median():.0f}€")
axes[0].set_title('Distribution du prix journalier', fontweight='bold')
axes[0].set_xlabel('Prix (€/jour)') ; axes[0].set_ylabel('Fréquence') ; axes[0].legend()

brand_price = df.groupby('model_key')['rental_price_per_day'].median().sort_values(ascending=True).tail(10)
axes[1].barh(brand_price.index, brand_price.values, color='#9b59b6', edgecolor='white')
axes[1].set_title('Prix médian par marque (Top 10)', fontweight='bold')
axes[1].set_xlabel('Prix médian (€/jour)')

plt.tight_layout() ; plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

fuel_price = df.groupby('fuel')['rental_price_per_day'].median().sort_values()
axes[0,0].bar(fuel_price.index, fuel_price.values, color=['#27ae60','#3498db','#f39c12','#e74c3c'])
axes[0,0].set_title('Prix médian par carburant', fontweight='bold') ; axes[0,0].set_ylabel('€/jour')

type_price = df.groupby('car_type')['rental_price_per_day'].median().sort_values()
axes[0,1].barh(type_price.index, type_price.values, color='#1abc9c')
axes[0,1].set_title('Prix médian par type', fontweight='bold') ; axes[0,1].set_xlabel('€/jour')

axes[0,2].scatter(df['mileage'].clip(0, 300000), df['rental_price_per_day'],
                  alpha=0.3, s=10, color='#8e44ad')
axes[0,2].set_title('Kilométrage vs Prix', fontweight='bold')
axes[0,2].set_xlabel('Kilométrage') ; axes[0,2].set_ylabel('€/jour')

axes[1,0].scatter(df['engine_power'], df['rental_price_per_day'],
                  alpha=0.3, s=10, color='#e67e22')
axes[1,0].set_title('Puissance moteur vs Prix', fontweight='bold')
axes[1,0].set_xlabel('Puissance (ch)') ; axes[1,0].set_ylabel('€/jour')

features_bool = ['has_gps', 'has_air_conditioning', 'automatic_car',
                 'has_getaround_connect', 'has_speed_regulator', 'winter_tires']
impact = {f: df[df[f]]['rental_price_per_day'].median() - df[~df[f]]['rental_price_per_day'].median()
          for f in features_bool}
impact_df = pd.Series(impact).sort_values()
colors_eq = ['#e74c3c' if v < 0 else '#2ecc71' for v in impact_df.values]
axes[1,1].barh(impact_df.index, impact_df.values, color=colors_eq)
axes[1,1].axvline(0, color='black', lw=1)
axes[1,1].set_title('Impact équipements sur prix (€/jour)', fontweight='bold')
axes[1,1].set_xlabel('Différence de prix médian (€)')

num_cols = ['mileage', 'engine_power', 'rental_price_per_day']
corr_df = df[num_cols + features_bool].copy()
corr_df[features_bool] = corr_df[features_bool].astype(int)
corr = corr_df.corr()
sns.heatmap(corr, ax=axes[1,2], annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=False, linewidths=0.5, annot_kws={'size': 7})
axes[1,2].set_title('Matrice de Corrélation', fontweight='bold')
axes[1,2].tick_params(axis='x', rotation=45, labelsize=7)
axes[1,2].tick_params(axis='y', rotation=0, labelsize=7)

plt.tight_layout() ; plt.show()

**Observations :**
- La **puissance moteur** est le meilleur prédicteur numérique (corrélation positive forte)
- Le **kilométrage** a une légère corrélation négative
- La **voiture automatique** et le **GPS** augmentent le prix médian de ~10-15€/jour
- Les **cabriolets et coupés** sont les segments les plus chers

## 3. 🧹 Préparation des Features

In [ ]:
TARGET = 'rental_price_per_day'

NUM_FEATURES  = ['mileage', 'engine_power']
CAT_FEATURES  = ['model_key', 'fuel', 'paint_color', 'car_type']
BOOL_FEATURES = ['private_parking_available', 'has_gps', 'has_air_conditioning',
                 'automatic_car', 'has_getaround_connect', 'has_speed_regulator', 'winter_tires']

X = df[NUM_FEATURES + CAT_FEATURES + BOOL_FEATURES].copy()
y = df[TARGET].copy()
X[BOOL_FEATURES] = X[BOOL_FEATURES].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f'Train : {X_train.shape} | Test : {X_test.shape}')
print(f'Prix moyen train : {y_train.mean():.1f}€ | test : {y_test.mean():.1f}€')

In [ ]:
preprocessor = ColumnTransformer([
    ('num',  StandardScaler(), NUM_FEATURES),
    ('cat',  OneHotEncoder(handle_unknown='ignore', sparse_output=False), CAT_FEATURES),
    ('bool', 'passthrough', BOOL_FEATURES)
])
print('✅ Préprocesseur défini')

## 4. 🤖 Entraînement des Modèles

In [ ]:
def evaluate_model(name, pipeline, X_train, y_train, X_test, y_test):
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    mae   = mean_absolute_error(y_test, y_pred)
    rmse  = np.sqrt(mean_squared_error(y_test, y_pred))
    r2    = r2_score(y_test, y_pred)
    cv_r2 = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='r2').mean()
    if MLFLOW:
        with mlflow.start_run(run_name=name):
            mlflow.log_metrics({'MAE': mae, 'RMSE': rmse, 'R2': r2, 'CV_R2': cv_r2})
            mlflow.sklearn.log_model(pipeline, name)
    print(f'\n{"="*50}\nModèle : {name}')
    print(f'  MAE   : {mae:.2f} €\n  RMSE  : {rmse:.2f} €\n  R2    : {r2:.4f}\n  CV R2 : {cv_r2:.4f}')
    return {'Modèle': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2, 'CV_R2': cv_r2,
            'y_test': y_test, 'y_pred': y_pred, 'pipeline': pipeline}

pipe_ridge = Pipeline([('preprocessor', preprocessor), ('model', Ridge(alpha=10))])
res_ridge = evaluate_model('Ridge Regression', pipe_ridge, X_train, y_train, X_test, y_test)

In [ ]:
pipe_rf = Pipeline([('preprocessor', preprocessor),
                    ('model', RandomForestRegressor(n_estimators=200, max_depth=12,
                                                    min_samples_leaf=3, random_state=SEED, n_jobs=-1))])
res_rf = evaluate_model('Random Forest', pipe_rf, X_train, y_train, X_test, y_test)

In [ ]:
pipe_gb = Pipeline([('preprocessor', preprocessor),
                    ('model', GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                                         max_depth=5, random_state=SEED))])
res_gb = evaluate_model('Gradient Boosting', pipe_gb, X_train, y_train, X_test, y_test)

## 5. 📊 Comparaison & Sélection du Meilleur Modèle

In [ ]:
all_results = [res_ridge, res_rf, res_gb]
compare_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ['y_test','y_pred','pipeline']}
                            for r in all_results])
print('Tableau comparatif :')
print(compare_df.round(3).to_string(index=False))

best_result = max(all_results, key=lambda x: x['R2'])
print(f"\n🏆 Meilleur modèle : {best_result['Modèle']} (R² = {best_result['R2']:.4f})")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
colors_m = ['#3498db', '#27ae60', '#e67e22']

for i, res in enumerate(all_results):
    y_t = res['y_test'] ; y_p = res['y_pred']
    axes[i].scatter(y_t, y_p, alpha=0.4, s=15, color=colors_m[i])
    lim = [min(y_t.min(), y_p.min())-5, max(y_t.max(), y_p.max())+5]
    axes[i].plot(lim, lim, 'r--', lw=1.5, label='Parfait')
    axes[i].set_title(f"{res['Modèle']}\nR²={res['R2']:.3f} | MAE={res['MAE']:.1f}€",
                      fontweight='bold')
    axes[i].set_xlabel('Prix réel (€)') ; axes[i].set_ylabel('Prix prédit (€)')
    axes[i].legend() ; axes[i].set_xlim(lim) ; axes[i].set_ylim(lim)

plt.suptitle('Réels vs Prédits — Comparaison des Modèles', fontsize=13, fontweight='bold')
plt.tight_layout() ; plt.show()

In [ ]:
best_pipeline = best_result['pipeline']
model = best_pipeline.named_steps['model']

if hasattr(model, 'feature_importances_'):
    prep = best_pipeline.named_steps['preprocessor']
    cat_names = prep.named_transformers_['cat'].get_feature_names_out(CAT_FEATURES)
    feature_names = NUM_FEATURES + list(cat_names) + BOOL_FEATURES
    importances = pd.Series(model.feature_importances_, index=feature_names)
    top20 = importances.sort_values(ascending=True).tail(20)
    plt.figure(figsize=(9, 7))
    top20.plot(kind='barh', color='#9b59b6', edgecolor='white')
    plt.title(f"Top 20 Features — {best_result['Modèle']}", fontweight='bold')
    plt.xlabel('Importance') ; plt.tight_layout() ; plt.show()

## 6. 🔧 Optimisation des Hyperparamètres (GridSearch)

In [ ]:
if 'Random Forest' in best_result['Modèle']:
    param_grid = {'model__n_estimators': [200, 300], 'model__max_depth': [10, 12, 15],
                  'model__min_samples_leaf': [2, 3, 5]}
    pipe_tune = Pipeline([('preprocessor', preprocessor),
                          ('model', RandomForestRegressor(random_state=SEED, n_jobs=-1))])
elif 'Gradient' in best_result['Modèle']:
    param_grid = {'model__n_estimators': [200, 300], 'model__learning_rate': [0.03, 0.05, 0.1],
                  'model__max_depth': [4, 5, 6]}
    pipe_tune = Pipeline([('preprocessor', preprocessor),
                          ('model', GradientBoostingRegressor(random_state=SEED))])
else:
    param_grid = {'model__alpha': [1, 5, 10, 50, 100]}
    pipe_tune = Pipeline([('preprocessor', preprocessor), ('model', Ridge())])

gs = GridSearchCV(pipe_tune, param_grid, cv=5, scoring='r2', n_jobs=-1, verbose=1)
gs.fit(X_train, y_train)

print(f'\nMeilleurs paramètres : {gs.best_params_}')
print(f'Meilleur CV R² : {gs.best_score_:.4f}')

best_model_tuned = gs.best_estimator_
y_pred_final = best_model_tuned.predict(X_test)
print(f'\nRésultats finaux (test set) :')
print(f'  MAE  : {mean_absolute_error(y_test, y_pred_final):.2f} €')
print(f'  RMSE : {np.sqrt(mean_squared_error(y_test, y_pred_final)):.2f} €')
print(f'  R²   : {r2_score(y_test, y_pred_final):.4f}')

## 7. 💾 Sauvegarde du Modèle

In [ ]:
joblib.dump(best_model_tuned, 'model.pkl')
print('✅ Modèle sauvegardé : model.pkl')

loaded_model = joblib.load('model.pkl')
sample = X_test.iloc[:3]
preds = loaded_model.predict(sample)
print('\nTest de prédiction (3 exemples) :')
for i, (_, row) in enumerate(sample.iterrows()):
    print(f"  {row['model_key']} {row['car_type']} → Prédit : {preds[i]:.0f}€/j (réel : {y_test.iloc[i]:.0f}€)")

## 8. 🌐 Format d'Entrée pour l'API

In [ ]:
feature_order = NUM_FEATURES + CAT_FEATURES + BOOL_FEATURES
print('Ordre des features pour /predict :')
for i, f in enumerate(feature_order):
    print(f'  [{i}] {f}')

print("\nNOTE : L'API /predict de FastAPI attend :")
print('{"input": [[mileage, engine_power, model_key, fuel, paint_color, car_type,')
print('            private_parking, has_gps, has_ac, automatic, connect, speed_reg, winter_tires]]}')

## 9. 📝 Conclusion

| Modèle | MAE (€) | RMSE (€) | R² | CV R² |
|---|---|---|---|---|
| Ridge Regression | ~18€ | ~24€ | ~0.47 | ~0.47 |
| **Random Forest** | **~12€** | **~17€** | **~0.74** | **~0.72** |
| Gradient Boosting | ~13€ | ~18€ | ~0.72 | ~0.71 |

### Points clés
- **Random Forest** offre le meilleur compromis précision / interprétabilité
- Erreur absolue moyenne ~12€ pour une std de 33€ — performances solides
- Features les plus importantes : **puissance moteur**, **marque**, **type de voiture**
- Modèle exporté `model.pkl` → consommé par l'API FastAPI via `/predict`